# 모델A 예측값 생성

예측 기간의 매 거래일 t마다, t일 종가까지의 데이터만 사용해 t+H거래일 종가를 예측한다.
결과는 B 학습용 CSV(`modelA_predictions_H{H}.csv`)로 저장한다.

- 메인 모델: Chronos-2 (가격 + 거래량 + 매크로)
- 비교용: Chronos-2 (가격만), TimesFM-2.5 (가격만), Kronos-base (시가·고가·저가·종가·거래량)
- 매크로: 미국 재무부 10년물 금리, 10년물−2년물 금리차, 10년 기대인플레이션(명목−실질), QQQ 20일 변동성, 유가(USO), 달러(UUP) (FRED·VIX는 약관 문제로 미사용)

런타임 유형은 T4 GPU.

## 0. 설치

In [ ]:
!pip install -q -U "chronos-forecasting>=2.0" "timesfm[torch]>=2.0" yfinance

## 1. 설정
- `CTX`: 모델에 넣는 과거 길이(거래일). 가격과 매크로가 같은 길이로 들어간다. 252 = 약 1년, 378 = 약 1년 반.

In [ ]:
TICKERS     = ["NVDA", "AMD", "MU", "TSM", "AVGO", "MRVL",   # 반도체
               "MSFT", "GOOG", "AMZN", "META"]               # 하이퍼스케일러
H           = 3                  # t+H거래일 예측
PRED_START  = "2026-04-01"       # 예측 시작일
PRED_END    = "2026-10-31"       # 예측 종료일 (데이터가 있는 날까지만 예측)
CTX         = 378                # 과거 입력 길이 (252~378)
USE_MACRO   = True
MKT_TICKER  = "QQQ"              # 시장 변동성 계산용
OIL_TICKER  = "USO"              # 유가 (원유 ETF)
USD_TICKER  = "UUP"              # 달러 (달러 인덱스 ETF)
RUN_COMPARE = False              # 비교용 모델(Chronos-2 가격만, TimesFM-2.5)도 돌릴지
RUN_KRONOS  = False              # 비교용 Kronos (GPU 필요)
KR_CTX      = 40                 # Kronos 입력 길이 (논문 일봉 실험: 40일)
KR_T        = 0.6                # Kronos 샘플링 온도 (논문 설정)
KR_SAMPLES  = 10                 # Kronos 샘플 수 (논문 설정)

## 2. 데이터
- 가격: yfinance 수정주가. 미국 장중에 실행하면 당일 미완성 봉은 제외.
- 재무부 금리: 사이트에서 연도별 명목·실질 금리 CSV를 받는다. 기대인플레이션 = 명목 10년 − 실질 10년.
- 유가·달러: yfinance의 원유 ETF(USO), 달러 ETF(UUP) 종가.
- 재무부 사이트가 코랩에서 막히면, 재무부 사이트에서 받은 명목·실질 금리 CSV를 파일 패널에 올리면 그 파일을 쓴다(파일 이름에 `yield` 또는 `treasury` 포함).
- 매크로는 휴일 빈칸을 직전 값으로 채우고 하루 늦춰 붙인다(발표 시차로 인한 미래 정보 혼입 방지).

In [ ]:
import io, os, time, warnings
import numpy as np
import pandas as pd
import requests
import yfinance as yf
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
START = str((pd.Timestamp(PRED_START) - pd.DateOffset(days=int(CTX * 1.6) + 60)).date())   # CTX + 여유분
TIMES = {}
UA = {"User-Agent": "Mozilla/5.0"}
t0 = time.perf_counter()

def load_prices(ticker, start):
    try:
        df = yf.Ticker(ticker).history(start=start, auto_adjust=True)
    except Exception as e:
        print(f"{ticker} yfinance 오류:", e)
        df = pd.DataFrame()
    if df.empty:
        print(f"{ticker}: yfinance 실패 → {ticker}.csv 사용")
        df = pd.read_csv(f"{ticker}.csv", index_col=0, parse_dates=True)
    idx = pd.to_datetime(df.index)
    if idx.tz is not None:
        idx = idx.tz_localize(None)
    df.index = idx.normalize()
    df = df[["Open", "High", "Low", "Close", "Volume"]].astype(float).dropna()
    df.columns = [c.lower() for c in df.columns]
    now_ny = pd.Timestamp.now(tz="America/New_York")
    if df.index[-1].date() == now_ny.date() and now_ny.hour < 16:
        df = df.iloc[:-1]
    return df

def read_treasury(src):
    # 명목 금리표(2 Yr 열 있음)와 실질 금리표(2 Yr 열 없음) 모두 읽음
    df = pd.read_csv(src)
    cols = {c.strip().lower(): c for c in df.columns}
    out = pd.DataFrame(index=pd.to_datetime(df[df.columns[0]]))
    for key, name in [("10 yr", "y10"), ("2 yr", "y2")]:
        if key in cols:
            out[name] = pd.to_numeric(df[cols[key]], errors="coerce").values
    return out

def treasury_url(kind, y):
    return ("https://home.treasury.gov/resource-center/data-chart-center/interest-rates/"
            f"daily-treasury-rates.csv/{y}/all?type={kind}&field_tdr_date_value={y}&page&_format=csv")

def load_treasury():
    # 반환: 명목 10년·2년, 실질 10년 (일별)
    local = sorted(f for f in os.listdir(".") if f.lower().endswith(".csv")
                   and ("yield" in f.lower() or "treasury" in f.lower()))
    nominal, real = [], []
    if local:
        print("재무부 금리: 업로드 파일 사용 →", ", ".join(local))
        for f in local:
            t = read_treasury(f)
            (nominal if "y2" in t else real).append(t)
    else:
        for y in range(int(START[:4]), pd.Timestamp.today().year + 1):
            for kind, bucket in [("daily_treasury_yield_curve", nominal),
                                 ("daily_treasury_real_yield_curve", real)]:
                r = requests.get(treasury_url(kind, y), headers=UA, timeout=30)
                r.raise_for_status()
                bucket.append(read_treasury(io.StringIO(r.text)))
        print(f"재무부 금리: 명목·실질 각 {len(nominal)}개 연도 다운로드")
    def merge(frames):
        d = pd.concat(frames).sort_index()
        return d[~d.index.duplicated(keep="last")]
    n = merge(nominal)
    out = pd.DataFrame({"UST10Y": n["y10"], "SPREAD_10Y2Y": n["y10"] - n["y2"]})
    if real:
        r10 = merge(real)["y10"].reindex(out.index.union(merge(real).index))
        out = out.reindex(r10.index)
        out["BEI10Y"] = out["UST10Y"] - r10        # 기대인플레이션 = 명목 − 실질
    return out

def load_market_vol(ticker):
    c = load_prices(ticker, START)["close"]
    rv = np.log(c).diff().rolling(20).std() * np.sqrt(252) * 100
    return rv.rename("MKT_RV20").to_frame()

def load_etf(ticker, name):
    return load_prices(ticker, START)["close"].rename(name).to_frame()

MACRO = {}
parts = []
if USE_MACRO:
    try:
        t = load_treasury()
        parts.append(t)
        MACRO.update({"UST10Y": "10년물 금리", "SPREAD_10Y2Y": "10년-2년 금리차"})
        if "BEI10Y" in t:
            MACRO["BEI10Y"] = "10년 기대인플레이션(명목−실질)"
        else:
            print("재무부 실질금리 없음 → 물가 변수 제외")
    except Exception as e:
        print("재무부 금리 실패 → 제외:", repr(e))
    for fn, args, key, desc in [(load_market_vol, (MKT_TICKER,), "MKT_RV20", f"{MKT_TICKER} 20일 변동성"),
                                (load_etf, (OIL_TICKER, "OIL"), "OIL", f"유가({OIL_TICKER})"),
                                (load_etf, (USD_TICKER, "USD"), "USD", f"달러({USD_TICKER})")]:
        try:
            parts.append(fn(*args))
            MACRO[key] = desc
        except Exception as e:
            print(f"{desc} 실패 → 제외:", repr(e))
macro_raw = pd.concat(parts, axis=1) if parts else None

def build(ticker):
    px = load_prices(ticker, START)
    if macro_raw is not None:
        m = (macro_raw.reindex(macro_raw.index.union(px.index)).ffill()
                      .reindex(px.index).shift(1))
        px = px.join(m)
    px["log_volume"] = np.log1p(px["volume"])
    return px.dropna()

DATA = {tk: build(tk) for tk in TICKERS}
TIMES["데이터 다운로드"] = time.perf_counter() - t0

print("매크로:", ", ".join(f"{k}({v})" for k, v in MACRO.items()) or "없음")
for tk, d in DATA.items():
    print(f"{tk}: {d.index[0].date()} ~ {d.index[-1].date()} ({len(d)}거래일)")

## 3. 예측 시점 구성
각 종목에서 `PRED_START` ~ `PRED_END` 사이의 모든 거래일이 예측 시점이다. 마지막 H일은 아직 실제값이 없어 잔차가 비어 있다.

In [ ]:
ORIGINS = []   # (종목, 위치)
for tk, d in DATA.items():
    mask = (d.index >= pd.Timestamp(PRED_START)) & (d.index <= pd.Timestamp(PRED_END))
    pos = np.flatnonzero(mask)
    pos = pos[pos >= CTX - 1]
    if len(pos) and pos[0] > np.flatnonzero(mask)[0]:
        print(f"주의: {tk}는 과거 데이터가 부족해 일부 시점을 건너뜀")
    ORIGINS += [(tk, int(p)) for p in pos]

base = np.array([DATA[tk]["close"].iloc[p] for tk, p in ORIGINS])
real = np.array([DATA[tk]["close"].iloc[p + H] if p + H < len(DATA[tk]) else np.nan for tk, p in ORIGINS])
dates = pd.DatetimeIndex([DATA[tk].index[p] for tk, p in ORIGINS])
tdates = [DATA[tk].index[p + H] if p + H < len(DATA[tk]) else pd.NaT for tk, p in ORIGINS]
real_ret = real / base - 1
n_days = dates.nunique()
print(f"예측 시점: {len(ORIGINS)}개 ({len(TICKERS)}종목 × {n_days}거래일), {dates.min().date()} ~ {dates.max().date()}")

## 4. Chronos-2

In [ ]:
import torch
from chronos import Chronos2Pipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("장치:", DEVICE)

t0 = time.perf_counter()
pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=DEVICE)
TIMES["Chronos-2 로드"] = time.perf_counter() - t0

def make_input(tk, p, cov_cols):
    d = DATA[tk]
    sl = slice(p - CTX + 1, p + 1)
    item = {"target": d["close"].values[sl].astype(np.float32)}
    if cov_cols:
        item["past_covariates"] = {c: d[c].values[sl].astype(np.float32) for c in cov_cols}
    return item

def chronos_forecast(cov_cols):
    items = [make_input(tk, p, cov_cols) for tk, p in ORIGINS]
    q, _ = pipeline.predict_quantiles(items, prediction_length=H, quantile_levels=[0.1, 0.5, 0.9])
    return np.sort(np.stack([x[0].numpy() for x in q]), axis=-1)   # (시점, H, 3)

def timed(fn, *args):
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t = time.perf_counter()
    out = fn(*args)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    return out, time.perf_counter() - t

PRED = {}
MAIN = "Chronos-2 | price+volume+macro" if MACRO else "Chronos-2 | price+volume"
PRED[MAIN], sec = timed(chronos_forecast, ["log_volume"] + list(MACRO))
TIMES[MAIN] = sec
if RUN_COMPARE:
    PRED["Chronos-2 | price"], sec = timed(chronos_forecast, [])
    TIMES["Chronos-2 | price"] = sec
for k in PRED:
    print(f"{k}: {TIMES[k]:.1f}초")

## 5. (비교용) TimesFM-2.5

In [ ]:
if RUN_COMPARE:
    try:
        import timesfm
        t0 = time.perf_counter()
        tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch", torch_compile=False)
        tfm.compile(timesfm.ForecastConfig(
            max_context=CTX, max_horizon=128, per_core_batch_size=32,
            normalize_inputs=True, use_continuous_quantile_head=True,
            force_flip_invariance=True, infer_is_positive=True, fix_quantile_crossing=True))
        TIMES["TimesFM-2.5 로드"] = time.perf_counter() - t0

        def timesfm_forecast():
            inputs = [DATA[tk]["close"].values[p - CTX + 1 : p + 1].astype(np.float32) for tk, p in ORIGINS]
            _, q = tfm.forecast(horizon=H, inputs=inputs)
            return np.sort(q[:, :, [1, 5, 9]], axis=-1)

        PRED["TimesFM-2.5 | price"], sec = timed(timesfm_forecast)
        TIMES["TimesFM-2.5 | price"] = sec
        print(f"TimesFM-2.5: {sec:.1f}초")
    except Exception as e:
        print("TimesFM-2.5 오류 → 건너뜀:", repr(e))

## 5-2. (비교용) Kronos-base
금융 전용 모델. 입력은 시가·고가·저가·종가·거래량(매크로 불가). 설정은 논문의 일봉 실험(과거 40일, 온도 0.6, 샘플 10개)을 따른다.
사전학습 데이터가 2024년 6월까지라 이번 예측 기간(2026년)과 겹치지 않는다.

In [ ]:
if RUN_KRONOS and DEVICE != "cuda":
    print("GPU가 없어 Kronos는 건너뜀")
elif RUN_KRONOS:
    try:
        import sys, subprocess
        KRONOS_DIR = os.path.abspath("Kronos")
        if not os.path.exists(KRONOS_DIR):
            subprocess.run(["git", "clone", "-q", "--depth", "1",
                            "https://github.com/shiyu-coder/Kronos.git", KRONOS_DIR], check=True)
        sys.path.insert(0, KRONOS_DIR)
        from model import Kronos, KronosTokenizer, KronosPredictor

        t0 = time.perf_counter()
        kr = KronosPredictor(Kronos.from_pretrained("NeoQuasar/Kronos-base"),
                             KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base"),
                             device="cuda:0", max_context=512)
        TIMES["Kronos 로드"] = time.perf_counter() - t0
        OHLCV = ["open", "high", "low", "close", "volume"]

        def kronos_forecast(chunk=64):
            out = []
            for i in range(0, len(ORIGINS), chunk):
                dfs, xts, yts = [], [], []
                for tk, p in ORIGINS[i:i + chunk]:
                    d = DATA[tk]
                    hist = d.iloc[p - KR_CTX + 1 : p + 1]
                    fut = d.index[p + 1 : p + 1 + H]                  # 다음 거래일 날짜만 사용
                    if len(fut) < H:
                        fut = pd.bdate_range(hist.index[-1] + pd.Timedelta(days=1), periods=H)
                    dfs.append(hist[OHLCV].reset_index(drop=True))
                    xts.append(pd.Series(hist.index))
                    yts.append(pd.Series(fut))
                preds = kr.predict_batch(dfs, xts, yts, pred_len=H, T=KR_T, top_p=0.9,
                                         sample_count=KR_SAMPLES, verbose=False)
                out += [x["close"].values for x in preds]
            close = np.array(out)                                     # (시점, H) 샘플 평균
            return np.repeat(close[:, :, None], 3, axis=2)            # 분위수 없음: 세 칸 모두 평균값

        torch.manual_seed(0)
        PRED["Kronos-base | OHLCV"], sec = timed(kronos_forecast)
        TIMES["Kronos-base | OHLCV"] = sec
        print(f"Kronos: {sec:.1f}초")
    except Exception as e:
        print("Kronos 오류 → 건너뜀:", repr(e))

## 6. 평가
- IC: 날짜마다 종목들 사이에서 예측 수익률과 실제 수익률의 순위 상관을 구해 평균. t값은 예측 구간이 겹치지 않도록 H일 간격 날짜만으로 계산.
- 방향 정확도, 수익률 MAE, 80% 구간 적중률은 전체 시점 기준.

In [ ]:
from scipy.stats import spearmanr

ok = ~np.isnan(real_ret)

def daily_ic(pred_ret):
    df = pd.DataFrame({"date": dates[ok], "p": pred_ret[ok], "r": real_ret[ok]})
    ic = pd.Series({d: spearmanr(g["p"], g["r"])[0] for d, g in df.groupby("date") if len(g) >= 3})
    return ic.dropna().sort_index()

def evaluate(f):
    pred_ret = f[:, -1, 1] / base - 1
    ic = daily_ic(pred_ret)
    ic_nov = ic.iloc[::H]
    return {
        "평균 IC": ic.mean(),
        "IC t값(비중첩)": ic_nov.mean() / ic_nov.std() * np.sqrt(len(ic_nov)) if len(ic_nov) > 2 else np.nan,
        "방향 정확도": np.mean(np.sign(pred_ret[ok]) == np.sign(real_ret[ok])),
        "수익률 MAE": np.mean(np.abs(pred_ret[ok] - real_ret[ok])),
        "80% 구간 적중률": np.mean((real[ok] >= f[ok, -1, 0]) & (real[ok] <= f[ok, -1, 2])) if (f[:, -1, 0] != f[:, -1, 2]).any() else np.nan,
    }

rows = {k: evaluate(f) for k, f in PRED.items()}
rows["Baseline | zero return"] = {"수익률 MAE": np.mean(np.abs(real_ret[ok]))}
rows["Baseline | always up"] = {"방향 정확도": np.mean(real_ret[ok] > 0)}
summary = pd.DataFrame(rows).T
print(f"평가 시점 {ok.sum()}개, 날짜 {dates[ok].nunique()}일. |t값| < 2면 우연과 구분하기 어려움.")
display(summary.round(3))

ic = daily_ic(PRED[MAIN][:, -1, 1] / base - 1)
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(ic.index, ic.values, width=1.0, color="lightgray")
ax.plot(ic.index, ic.rolling(20).mean(), color="tab:blue", label="20-day mean")
ax.axhline(0, color="black", lw=0.5)
ax.set_title(f"{MAIN}: daily cross-sectional IC ({H}-day return)")
ax.legend()
plt.show()

## 7. 저장
열 설명
- `date`: 예측 시점 t, `target_date`: t+H거래일
- `real_ret`: 실제 H일 수익률, `pred_ret`: 메인 모델 예측 수익률(중앙값), `pred_q10`/`pred_q90`: 10%/90% 분위수 수익률
- `residual`: real_ret − pred_ret (B의 타깃)
- `pred_ret | <모델>`: 비교용 모델 예측

In [ ]:
f = PRED[MAIN]
out = pd.DataFrame({
    "date": dates, "ticker": [tk for tk, _ in ORIGINS], "target_date": tdates,
    "close": base, "real_ret": real_ret,
    "pred_ret": f[:, -1, 1] / base - 1,
    "pred_q10": f[:, -1, 0] / base - 1,
    "pred_q90": f[:, -1, 2] / base - 1,
})
out["residual"] = out["real_ret"] - out["pred_ret"]
for k, g in PRED.items():
    if k != MAIN:
        out[f"pred_ret | {k}"] = g[:, -1, 1] / base - 1
out = out.sort_values(["date", "ticker"]).reset_index(drop=True)

fname = f"modelA_predictions_H{H}.csv"
out.to_csv(fname, index=False)
display(pd.Series(TIMES, name="초").round(2).to_frame())
print(f"저장: {fname} ({len(out)}행, 메인 모델: {MAIN})")
out.tail()